In [2]:
from pathlib import Path
import tensorflow as tf
from tensorflow.keras import mixed_precision 
import sys

# appending llm_components path to sys.path to easily import
sys.path.append('/kaggle/input/datasets/harshit1234g/axiomlm-utils')
import llm_components as lc

In [3]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'),
 PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

In [4]:
mixed_precision.set_global_policy('mixed_float16')

## Paths

In [5]:
# directories
directory = Path('/kaggle/input/datasets')
data_dir = directory / 'vadimkurochkin'/ 'wikitext-103' / 'wikitext-103'
utils_dir = directory / 'harshit1234g' / 'axiomlm-utils'

# dataset paths
train_path = data_dir / 'wiki.train.tokens'
valid_path = data_dir / 'wiki.valid.tokens'
test_path = data_dir / 'wiki.test.tokens'

# tokenizer
tokenizer_path =  utils_dir / 'sp_tokenizer.model'

# Path for checkpoint directory, the reason for 2 checkpoint directories is because 
# kaggle reads from input directory, but saves in working directory
checkpoint_restore_dir = utils_dir / 'checkpoints'
checkpoint_save_dir = Path('/kaggle/working/checkpoints')

## Hyper Parameters

In [6]:
SEQUENCE_LEN = 512      # Context size
SHIFT = SEQUENCE_LEN    # using shift = seq_len because the dataset is quite large
BATCH_SIZE = 64         # previously used 128 batch size, but got OOM
N_EMBEDS = 512
N_HEADS = 8
N_BLOCKS = 12
STEPS_PER_EPOCH = 3000

In [7]:
target_tokens = 900_000_000
token_per_step = BATCH_SIZE * SEQUENCE_LEN
token_per_chunk = STEPS_PER_EPOCH * token_per_step
total_steps = round(target_tokens // token_per_step, -3)
warmup_steps = int(total_steps * 0.05)  # 5%
print(f'Total target tokens: {target_tokens:3,}')
print(f'Steps per epoch: {STEPS_PER_EPOCH:3,}')
print(f'Token per step: {token_per_step:3,}')
print(f'Token per chunk/epoch: {token_per_chunk:3,}')
print(f'Total steps for cosine decay: {total_steps:3,}')
print(f'Warmup steps for cosine decay: {warmup_steps:3,}')

Total target tokens: 900,000,000
Steps per epoch: 3,000
Token per step: 32,768
Token per chunk/epoch: 98,304,000
Total steps for cosine decay: 27,000
Warmup steps for cosine decay: 1,350


## Loading Data

In [8]:
sp = lc.load_sp_tokenizer(str(tokenizer_path))
loader = lc.LMDatasetLoader(
    tokenizer= sp,
    shift= SHIFT,
    seq_len= SEQUENCE_LEN,
    batch_size= BATCH_SIZE,
    shuffle_buffer= 16_000
)

In [9]:
train_ds = loader.create(train_path, training= True)
valid_ds = loader.create(valid_path, training= False)
test_ds = loader.create(test_path, training= False)

I0000 00:00:1771333186.845148      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1771333186.851756      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [10]:
for item in train_ds.take(1):
    print(item)

(<tf.Tensor: shape=(64, 512), dtype=int32, numpy=
array([[ 3067,  6466,  3538, ...,  6832,  1277,  3535],
       [ 1643, 14778,   353, ...,  5896,  5777,   275],
       [ 1251,   267,   429, ...,   292,   297,   680],
       ...,
       [ 4874,  2022,   275, ...,   404,   264,  3445],
       [  285,  5758, 13106, ...,   267,  6542,   267],
       [  878,   297,  1208, ...,   330,   830,  6949]], dtype=int32)>, <tf.Tensor: shape=(64, 512), dtype=int32, numpy=
array([[ 6466,  3538,   264, ...,  1277,  3535,  2817],
       [14778,   353,  3504, ...,  5777,   275,  1213],
       [  267,   429,  5225, ...,   297,   680,   587],
       ...,
       [ 2022,   275,  1268, ...,   264,  3445,   285],
       [ 5758, 13106,   267, ...,  6542,   267,  1067],
       [  297,  1208,   376, ...,   830,  6949,  1410]], dtype=int32)>)


## Callbacks, Strategy & Vocab size

In [10]:
tensorboard_cb = tf.keras.callbacks.TensorBoard(
    log_dir= 'logs',
    histogram_freq= 1,
    embeddings_freq= 1
)

In [11]:
strategy = tf.distribute.MirroredStrategy()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


In [12]:
vocab_size = sp.get_piece_size()
vocab_size

16000

## Transformer Model

In [13]:
with strategy.scope():
    # creating model
    model = lc.GPT(
        vocab_size= vocab_size,
        seq_len= SEQUENCE_LEN,
        n_embeds= N_EMBEDS,
        n_heads= N_HEADS,
        n_blocks= N_BLOCKS
    )

    # lr schedule
    lr_schedule = lc.WarmupCosine(
        base_lr= 3e-4,
        warmup_steps= warmup_steps,
        total_steps= total_steps
    )

    # optimizer
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate= lr_schedule,
        weight_decay= 0.01,
        beta_2= 0.95,
        clipnorm= 1.0
    )
    
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits= True     # softmax is handled by loss function
    )

    # compiling
    model.compile(
        optimizer= optimizer,
        loss= loss_fn,
        metrics= [lc.Perplexity(pad_id = sp.pad_id())]
    )

    # # passing dummy input to build model
    dummy = tf.zeros((1, SEQUENCE_LEN), dtype= tf.int32)
    _ = model(dummy, training= False)

    # checkpoint logic
    checkpoint = tf.train.Checkpoint(
        model= model,
        optimizer= optimizer
    )

    latest_ch = tf.train.latest_checkpoint(checkpoint_restore_dir)
    if latest_ch:
        print('Restoring State from', latest_ch)
        
        optimizer.build(model.trainable_variables)
        checkpoint.restore(latest_ch).assert_existing_objects_matched()
        
        print('Step:', optimizer.iterations.numpy())
        print('LR:', lr_schedule(optimizer.iterations).numpy())
        print('Optimizer Variables:', len(optimizer.variables))    # must be larger than 200

    else:
        print('No Checkpoint found, random initialization.')

    manager = tf.train.CheckpointManager(
        checkpoint,
        checkpoint_save_dir,
        max_to_keep= 3
    )

Restoring State from /kaggle/input/datasets/harshit1234g/axiomlm-utils/checkpoints/ckpt-1
Step: 3000
LR: 0.0002972526
Optimizer Variables: 273


In [14]:
model.summary()

Model: "gpt"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 512, 512)          │     8,192,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (512, 512)             │       262,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_8             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_9             │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_10            │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_11            │ ?                      │     3,150,848 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_24          │ ?                      │         1,024 │
│ (LayerNormalization)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,265,344 (176.49 MB)

 Trainable params: 46,265,344 (176.49 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(
    train_ds, 
    steps_per_epoch= STEPS_PER_EPOCH,
    epochs= 1,
    validation_data= valid_ds,
    callbacks= [tensorboard_cb]
)

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Redu

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


3000/3000 ━━━━━━━━━━━━━━━━━━━━ 3609s 1s/step - loss: 3.8485 - perplexity: 47.0691 - val_loss: 3.0558 - val_perplexity: 32.8626


In [16]:
test_loss, test_perplexity = model.evaluate(test_ds)
print(f'{test_loss = }\n{test_perplexity = }')

8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 454ms/step - loss: 3.3987 - perplexity: 31.3882
test_loss = 3.087918281555176
test_perplexity = 32.262577056884766


In [17]:
manager.save()

'/kaggle/working/checkpoints/ckpt-2'

In [ ]:
model.save('AxiomLM-46M-Base.keras')

In [18]:
import subprocess

# creating zip of kaggle working directory to easily download it on my system
subprocess.run(['zip', '-r', 'working_dir.zip', '/kaggle/working'], stdout= subprocess.DEVNULL)

CompletedProcess(args=['zip', '-r', 'working_dir.zip', '/kaggle/working'], returncode=0)